# 🔍 Saeuron: *Autoencoders for Unlearning*

**Hands‑On Tutorial Notebook – KDD 2025 ‘Beyond Feature Attribution’**  
*Based on*: Cywinski*et al.* (2025) *SAeUron: Interpretable Concept Unlearning in Diffusion Models with Sparse Autoencoders* (ICML).  
*Official repo*: <https://github.com/cywinski/SAeUron>

<a target="_blank" href="https://colab.research.google.com/github/cxai-mechint-htutorial-kdd2025/cxai-mechint-htutorial-kdd2025.github.io/blob/main/notebooks/04_saeuron.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

---
### 🌟 What you will learn
1. **Discover** monosemantic latent concepts in a vision model using a Sparse Autoencoder (SAE).
2. **Intervene** on a generative model through one-neuron unlearning.

**Estimated runtime** (Colab, T4 GPU): ≈&nbsp;15 min using pretrained checkpoints.

---

SAeUron is a novel, interpretable method for unlearning unwanted concepts in text-to-image diffusion models, developed to address concerns about inadvertently generating harmful or undesirable content. The core of the method involves training Sparse Autoencoders (SAEs) in an unsupervised manner on the internal activations collected from multiple denoising timesteps within a diffusion model (e.g., Stable Diffusion). These SAEs are designed to learn sparse and semantically meaningful features that are interpretable and correspond to specific concepts, e.g. 'Dogs'. During the diffusion model's inference, SAeUron identifies features strongly associated with an unwanted concept using a calculated importance score. The influence of these selected concept-specific SAE features is then ablated by scaling their activations with a negative multiplier, effectively blocking the targeted content. This intervention is precise, avoiding modification of prompt embeddings or the base model's weights, thereby preserving overall model performance and offering enhanced transparency due to the direct interpretability of the modified features.

## 🗺️ Notebook Roadmap
1. **Setup & Dependencies**  
2. **Import and loading**  
3. **Image Generation & Basic Hooking**  
4. **Ablation and Unlearning**
5. **Computing Importance Scores**
6. **Exercises & References**

# 1 · Environment Setup
👉 **Run the cells below** on Colab (GPU runtime) to clone the repo and install all requirements. On a local machine, make sure you have CUDA‑enabled PyTorch ≥ 2.1.

In [ ]:
import os
current_dir = os.getcwd()
tutorial_repo = os.path.join(current_dir, 'SAeUron')
if not os.path.exists(tutorial_repo):
  !git clone https://github.com/cywinski/SAeUron.git
  !pip install -r SAeUron/requirements.txt
else:
  print('SAeUron installed')

  **If you are running this notebook in Colab, please restart your runtime now. This ensures that all installed packages and configurations take effect before proceeding.**

# 2 · Import and loading
👉 We load [stable-diffusion-v1-4](https://huggingface.co/CompVis/stable-diffusion-v1-4) (SD14), "a latent text-to-image diffusion model capable of generating photo-realistic images given any text input."

👉 We then load [the snapshot of a Sparse AutoeEncoder](https://huggingface.co/bcywinski/SAeUron_coco), trained by the authors of Saeuron. The SAE is trained by incorporating a sparsity penalty into the reconstruction
loss, ensuring that only a small fraction of latent neurons
activate for any given input.

In [ ]:
import sys
sys.path.append('SAeUron')
#
from tqdm.notebook import tqdm
import torch
import numpy as np
import matplotlib.pyplot as plt
from SAE.sae import Sae
from SAE.hooked_sd_noised_pipeline import HookedStableDiffusionPipeline
from SAE.unlearning_utils import compute_feature_importance
import utils.hooks as hooks
#
model_name = "CompVis/stable-diffusion-v1-4"
dtype = torch.float16
device = "cuda" if torch.cuda.is_available() else "cpu"
hub_name = "bcywinski/SAeUron_coco"
hookpoint = "unet.up_blocks.1.attentions.1"
num_inference_steps = 20

In [ ]:
image_generator = HookedStableDiffusionPipeline.from_pretrained(
    model_name,
    torch_dtype=dtype,
    safety_checker=None,
).to(device)

In [ ]:
sae = Sae.load_from_hub(hub_name, hookpoint=hookpoint, device=device).to(dtype)

# 3 · Image Generation & Basic Hooking
👉 We start by prompting SD14 in order to obtain an image  
👉 We use the hooks of SD14 to have dense activations go the SAE sparse hidden layer and back. We observe that the reconstruction is almost perfect.  
👉 We inspect the activations of SD14 and the SAE

In [ ]:
cat_prompt = "A realistic photography of a cat"
#
print("Image generated by SD14 for the prompt \'"+cat_prompt+"\'")
image_generator(
        prompt=cat_prompt,
        generator=torch.Generator(device=device).manual_seed(0),
        num_inference_steps=num_inference_steps,
)[0][0]

In [ ]:
reconstruction_hook = hooks.SAEReconstructHook(sae=sae)

In [ ]:
print("Image generated by SD14 *AND RECONSTRUCTED BY THE SAE* for the prompt \'"+cat_prompt+"\'")
img,cache = image_generator.run_with_hooks_and_cache(
        prompt=cat_prompt,
        generator=torch.Generator(device=device).manual_seed(0),
        num_inference_steps=num_inference_steps,
        position_hook_dict={hookpoint: reconstruction_hook},
        positions_to_cache=[hookpoint]
)
img[0][0]

In [ ]:
dense_activations = cache["output"][hookpoint].cpu()
plt.figure(figsize=(12,3))
plt.hist(dense_activations[0][-1].flatten())
plt.yscale('log');
plt.title('Dense activation for SD14');

In [ ]:
def sae_io(activations,num_iter=1,num_inference_steps=num_inference_steps):
  sae_latents = []
  with torch.no_grad():
      for i in range(num_iter):
          sae_in = activations[i].reshape(num_inference_steps, -1, sae.d_in)
          top_acts, top_indices = sae.encode(sae_in.to(sae.device))
          sae_out = torch.zeros(
              (top_acts.shape[0], sae.num_latents),
              device=sae.device,
              dtype=top_acts.dtype,
          ).scatter(-1, top_indices, top_acts)
          sae_out = sae_out.reshape(num_inference_steps, -1, sae.num_latents).cpu()
          sae_latents.append(sae_out.mean(1).to(dtype=torch.float16))#OCIO
  return torch.stack(sae_latents, dim=0)

In [ ]:
sparse_activations = sae_io(dense_activations)
plt.figure(figsize=(12,3))
plt.plot(sparse_activations[0][-1])
plt.title('Sparser activation for the SAE');

# 4 · Ablation & Unlearning  
👉 We start by ablating the 'magic numbered' latent 11627 in a cat image  
👉 We observe that other latents do not seem to impact cat images  
👉 We observe that latent 11627 seems to impact cat images but not other images

In [ ]:
def get_intervention_hook(latent_id):
    return hooks.SAEFeatureInterventionHook(
        sae=sae,
        feature_idx=latent_id,
        multiplier=-100.,
    )

In [ ]:
intervention_hook = get_intervention_hook(11627)

In [ ]:
print('Cat prompt + ablation of 11627 --> The cat is gone!')
image_generator.run_with_hooks(
        prompt=cat_prompt,
        generator=torch.Generator(device=device).manual_seed(0),
        num_inference_steps=num_inference_steps,
        position_hook_dict={hookpoint: intervention_hook},
    )[0]

In [ ]:
intervention_hook = get_intervention_hook(12345)
#
print('Cat prompt + ablation of another latent --> The cat is there!')
image_generator.run_with_hooks(
        prompt=cat_prompt,
        generator=torch.Generator(device=device).manual_seed(0),
        num_inference_steps=num_inference_steps,
        position_hook_dict={hookpoint: intervention_hook},
    )[0]

In [ ]:
church_prompt = "A realistic photography of a church"

print("Image generated by SD14 for the prompt \'"+church_prompt+"\'")
image_generator(
        prompt=church_prompt,
        generator=torch.Generator(device=device).manual_seed(0),
        num_inference_steps=num_inference_steps,
)[0][0]

In [ ]:
print("Image generated by SD14 *AND RECONSTRUCTED BY THE SAE* for the prompt \'"+church_prompt+"\'")
image_generator.run_with_hooks(
        prompt=church_prompt,
        generator=torch.Generator(device=device).manual_seed(0),
        num_inference_steps=num_inference_steps,
        position_hook_dict={hookpoint: reconstruction_hook},
)[0]

In [ ]:
intervention_hook = get_intervention_hook(11627)
#
print('Church prompt + ablation of 11627 (\'cat\') --> No impact')
image_generator.run_with_hooks(
        prompt=church_prompt,
        generator=torch.Generator(device=device).manual_seed(0),
        num_inference_steps=num_inference_steps,
        position_hook_dict={hookpoint: intervention_hook},
    )[0]

# 5 · Computing Importance Scores  
👉 Where does the 'magic numbered' latent 11627 come from?  
👉 Collection of SAE activations on a toy dataset  
👉 Computation of importance scores   
👉 Cross-ablation of selected latents and concepts

The authors of the Saeuron paper highlight 11627 as the most relevant 'cat' latent for SD14 and the current SAE.   
The value was computed through an original score function that measures the importance of each latent for each concept.   
This function is described in [Equation 4, page 4 in the Saeuron paper](https://arxiv.org/pdf/2501.18052), and the code is reported below.



````markdown
```python
def compute_feature_importance(style_latents_dict, target_style, timestep, epsilon=1e-8):
    if target_style not in style_latents_dict:
        raise ValueError(f"target_style '{target_style}' not found.")

    # Mean activation for the target style (shape: [num_features])
    latents_x = style_latents_dict[target_style][:, timestep, :].float()
    mean_x = latents_x.mean(dim=0)

    # All other styles
    other_styles = [s for s in style_latents_dict if s != target_style]
    if not other_styles:
        # If there's only one style, can't compare.
        return mean_x  # or torch.zeros_like(mean_x), depending on your needs

    # Mean activation for the combined "others"
    latents_others = torch.cat(
        [style_latents_dict[s][:, timestep, :].float() for s in other_styles], dim=0
    )
    mean_others = latents_others.mean(dim=0)

    # Denominators: total activation across all features
    total_x = mean_x.sum() + epsilon
    total_others = mean_others.sum() + epsilon

    # Proportions
    p_x = mean_x / total_x
    p_others = mean_others / total_others

    # Difference-based score
    scores = p_x - p_others

    return scores
```
````


### For a small set of concepts, we generate images through SD14 and collect SAE activations

In [ ]:
def collect_activations(item,num_iter=100):
    _,xdict = image_generator.run_with_cache(
        prompt="A realistic photography of a "+item,
        num_inference_steps=num_inference_steps,
        generator=torch.Generator(device=device).manual_seed(0),
        num_images_per_prompt=num_iter,
                positions_to_cache=[hookpoint],
                save_input = False,
                save_output = True,
    )
    activations = xdict["output"][hookpoint].cpu()
    return sae_io(activations,num_iter=num_iter,num_inference_steps=num_inference_steps)

In [ ]:
activation_dict = {}
for k in tqdm(['cat','spaceship','church']):
    activation_dict[k]=collect_activations(k,num_iter=15)

### We then compute the SCORE of every latent w.r.t our selected concepts

In [ ]:
scores = {k:compute_feature_importance(activation_dict,k,num_inference_steps-1) for k in activation_dict.keys()}
latents = {k:np.argmax(v.cpu().detach().numpy()) for (k,v) in scores.items()}

In [ ]:
plt.figure(figsize=(12,3))
for k,v in scores.items():
    plt.plot(v,label=k+' - MAX:'+str(latents[k]),alpha=.5);
plt.legend();
plt.xlabel('Latents in the SAE hidden layer')
plt.ylabel('Score (relevance for the concept)');

### We ablate the most important latent for each concept

In [ ]:
imgs = {}
subjects = list(latents.keys())
random_neuron = np.random.randint(20480)
ablations = ['no ablation']+[11627]+[random_neuron]+[latents[x] for x in subjects]
ablation_descr = ['no ablation']+['original cat neuron']+['random neuron '+str(random_neuron)]+['our own '+x+' neuron' for x in subjects]
for subject in subjects:
    for ablation in ablations:
        temp_hook = reconstruction_hook if ablation=='no ablation' else get_intervention_hook(ablation)
        imgs[(subject,ablation)] = image_generator.run_with_hooks(
            prompt="A realistic photography of a "+subject,
            generator=torch.Generator(device=device).manual_seed(0),
            num_inference_steps=num_inference_steps-1,
            position_hook_dict={hookpoint: temp_hook},
        )[0]

In [ ]:
fig, axes = plt.subplots(len(subjects), len(ablations), figsize=(4*len(ablations), 4*len(subjects)))
for i, subject in enumerate(subjects):
    for j, ablation in enumerate(ablations):
        ax = axes[i][j]
        ax.imshow(imgs[(subject, ablation)])
        ax.set_title(ablation_descr[j], fontsize=16)
        ax.axis('off')
plt.tight_layout()
plt.show()

The unlearning is not perfect, but there is a clear difference between 'matching' ablations (e.g., 'cat latent with cat image') wit respect to odd pairs. Bear in mind that our labelling was created via 15 images per concept.

# 6 · Exercises & Further Reading

1. In order to familiarise with SD14, **experiment with the parameters** *num_inference_steps* and *guidance_scale*.
2. Similarly, **explore the evolution of same-concept importance scores** by varying *timesteps* when calling *compute_feature_importance*.
3. **Write your own ablation hook** to intervene on more than one latent at a time.
4. **Collect importance scores at scale** by following the instructions of [the official repository](https://github.com/cywinski/SAeUron).

---
### 📑 References
- Nanda*et al.* (2025) *Steering Out-of-Distribution Generalization with Concept Ablation Fine-Tuning.* arXiv.
- Cunningham*et al.* (2024) *Sparse Autoencoders find highly interpretable features in language models.* ICLR.
- Cao*et al.* (2015) *Towards making systems forget with machine unlearning.* IEEE symposium on security and privacy  
- Farrell*et al.* (2024) *Applying sparse autoencoders to unlearn knowledge in language models.* arXiv.